# tinygpt — Colab training driver

This notebook is deliberately **thin**. It clones the repo, installs it, and calls into the package.
It contains no model code.

Consequence worth internalising: **anything that must run on the GPU has to be committed and pushed first.**
There is no way to test an uncommitted local edit from here.

Before running: **Runtime → Change runtime type → T4 GPU**.

See `docs/adr/0002` for why this design.

In [ ]:
# 1. Confirm we actually got a GPU. If this errors, the runtime type is wrong.
!nvidia-smi

In [ ]:
# 2. Mount Drive. Checkpoints go here because the VM's disk is destroyed
#    when the session ends (adr/0002).
from google.colab import drive

drive.mount('/content/drive')

CKPT_ROOT = '/content/drive/MyDrive/tinygpt/checkpoints'
!mkdir -p "$CKPT_ROOT"
print('checkpoints ->', CKPT_ROOT)

In [ ]:
# 3. Clone and install. Re-running this cell picks up new commits.
import os

REPO = 'https://github.com/jackwiencek/decoder-only-transformer.git'

if os.path.exists('/content/decoder-only-transformer'):
    !cd /content/decoder-only-transformer && git pull
else:
    !git clone $REPO /content/decoder-only-transformer

%cd /content/decoder-only-transformer
!pip install -q -e .

# Colab ships its own CUDA torch; do NOT reinstall torch here.

In [ ]:
# 4. Environment sanity checks — same tests that run on the laptop.
!python -m pytest -q

import torch

print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# 5. Experiment tracking (adr/0006). Paste the key from wandb.ai/authorize.
#    Skipping this cell is fine — the logger degrades to a no-op.
import wandb

wandb.login()

## Training

Not wired up yet — `src/tinygpt/` is empty by design (`docs/adr/0007`).
Once the data pipeline and training loop exist, this becomes roughly:

```python
!python -m tinygpt.data --prepare          # download + tokenize + cache to data/*.bin
!python -m tinygpt.train --config configs/base.yaml --checkpoint-dir "$CKPT_ROOT"
```